In [ ]:
import pandas as pd
import glob

# Busca todos os arquivos de ranking gerados pelas 20 batches
files = glob.glob("../experiments_results/predicted_ranking/top_30_birds_xgboost_batch_*.csv")

if not files:
    print("Nenhum arquivo encontrado")
else:
    # Junta todos os resultados
    df_massive = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    
    # Filtra apenas os candidatos acima de 0.7
    df_best_candidates = df_massive[df_massive['predicted_F1'] > 0.7].sort_values(by='predicted_F1', ascending=False)
    
    print(f"Análise dos candidatos finalizada!")
    print(f"Encontrados {len(df_best_candidates)} pipelines com F1 > 0.7")
    print("-" * 50)

    if len(df_best_candidates) > 0:
        # Identifica as colunas de Algoritmos (Flags)
        exclude = ['predicted_F1', 'predicted_model_size_log']
        flag_cols = [c for c in df_best_candidates.columns if '-' not in c and c not in exclude]
        
        # Conta a frequência dos algoritmos nos melhores candidatos
        counts = df_best_candidates[flag_cols].sum().sort_values(ascending=False)
        counts = counts[counts > 0] 
        
        print("Algoritmos mais frequentes nos melhores candidatos:")
        print(counts.to_string()) 
        
        # Salva os melhores candidatos para análise posterior 
        output_path = "../experiments_results/predicted_ranking/best_candidates_birds.csv"
        df_best_candidates.to_csv(output_path, index=False)
        
        # Exibe os algoritmos e parâmetros do melhor candidato
        print("\nDetalhes do melhor candidato:")
        top_1 = df_best_candidates.iloc[0]
        print(f"F1 Previsto: {top_1['predicted_F1']:.4f}")
        
        # Filtra apenas o que está ativo no Top 1
        ativos = top_1[(top_1 != -1.0) & (top_1 != 0.0)]
        for col, val in ativos.items():
            if col not in exclude:
                tipo = "Parâmetro" if '-' in col else "Algoritmo"
                print(f"   {tipo}: {col} = {val}")

Análise dos candidatos finalizada!
Encontrados 9 pipelines com F1 > 0.7
--------------------------------------------------
Algoritmos mais frequentes nos melhores candidatos:
feature preprocessing                        9.0
meka.classifiers.multilabel.MULAN.MLkNN      9.0
weka.classifiers.trees.RandomForest          7.0
sklearn.feature_selection.SelectFromModel    2.0
mlfs.br_relieff.BR_ReliefF                   2.0
sklearn.feature_selection.RFE                2.0
weka.classifiers.lazy.KStar                  2.0
mlfs.ppt_skb.PPT_SelectKBest                 1.0
mlfs.lrfs_adapted.PyIT_LRFS                  1.0
mlfs.br_skb.BR_SelectKBest                   1.0

Detalhes do melhor candidato:
F1 Previsto: 0.7389
   Algoritmo: feature preprocessing = 1.0
   Algoritmo: mlfs.br_relieff.BR_ReliefF = 1.0
   Parâmetro: mlfs.br_relieff.BR_ReliefF-n_features = 17.9875400969749
   Algoritmo: meka.classifiers.multilabel.MULAN.MLkNN = 1.0
   Parâmetro: meka.classifiers.multilabel.MULAN.MLkNN-numOfNeigh